## Libraries import

In [37]:
import os
from PIL import Image
from torch.utils.data import Dataset, Subset, DataLoader
from torchvision import transforms
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from tqdm import tqdm

## Dataset Class

In [38]:
class CelebADataset(Dataset):
    def __init__(self, label_file, image_dir, partition_file, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.samples = []
        self.partition_map = {}
        self.train_indices = []
        self.val_indices = []
        self.test_indices = []

        with open(partition_file, 'r') as f:
            for line in f:
                img_name, partition = line.strip().split()
                base_name = os.path.splitext(img_name)[0]
                self.partition_map[base_name] = int(partition)

        with open(label_file, 'r') as f:
            for line in f:
                img_name, label = line.strip().split()
                self.samples.append((img_name, int(label)))

        for idx, (img_name, _) in enumerate(self.samples):
            base_name = os.path.splitext(img_name)[0]
            split = self.partition_map[base_name]
            if split == 0:
                self.train_indices.append(idx)
            elif split == 1:
                self.val_indices.append(idx)
            elif split == 2:
                self.test_indices.append(idx)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, label = self.samples[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

## Data Loading & Transform

In [ ]:
base_path = '../AdvCelebA'
label_file = os.path.join(base_path, 'attack_CelebA.txt')
image_dir = os.path.join(base_path, 'images')
partition_file = os.path.join(base_path, 'list_eval_partition_no_overlap.txt')

transform = transforms.ToTensor()

dataset = CelebADataset(label_file, image_dir, partition_file, transform=transform)

train_dataset = Subset(dataset, dataset.train_indices)
val_dataset = Subset(dataset, dataset.val_indices)
test_dataset = Subset(dataset, dataset.test_indices)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(val_dataset, batch_size=32)

## Model (Simple CNN)

In [40]:
class BinaryClassifier(nn.Module):
    def __init__(self):
        super(BinaryClassifier, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # -> (16, 56, 56)
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)  # -> (32, 28, 28)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

## Training Loop

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BinaryClassifier().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 3

for epoch in range(epochs):
    model.train()
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1: 100%|██████████| 5072/5072 [06:52<00:00, 12.31it/s]


Epoch 1, Loss: 0.3906


Epoch 2: 100%|██████████| 5072/5072 [07:15<00:00, 11.66it/s]


Epoch 2, Loss: 0.2723


Epoch 3: 100%|██████████| 5072/5072 [07:15<00:00, 11.65it/s]

Epoch 3, Loss: 0.4195


In [42]:
torch.save(model.state_dict(), '../models/cnn_baseline.pth')

## Validation Accuracy

In [43]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        preds = torch.sigmoid(outputs).cpu().numpy() > 0.5
        all_preds.extend(preds.flatten())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"Validation Accuracy: {acc:.4f}")

Validation Accuracy: 0.8773
